# 01 — Data audit: Same 40, Different Burst

**Question:** Do players with similar official 40 times build speed differently, and do those differences translate to NFL performance?

Before calculating acceleration, establish what the files contain and whether the start of each sprint was captured. This notebook inventories the actual download; it does not assume raw column names or claim that the first recorded frame is the physical start.

Run cells from top to bottom in VS Code with a Python notebook kernel. The project must include `scripts/audit_data.py`. No results are embedded yet.


In [1]:
from pathlib import Path
import sys
import json
import importlib
import pandas as pd
import numpy as np
from IPython.display import display

# Works when VS Code starts the kernel in the project root or notebooks folder.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts" / "audit_data.py").is_file()), None)

if ROOT is None:
    raise FileNotFoundError("Open the nfl-big-data-bowl-2027 folder in VS Code first.")

OUTPUT = ROOT / "outputs" / "audit"
OUTPUT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "scripts"))
import audit_data
importlib.reload(audit_data)
print("Project:", ROOT)

Project: /Users/hunterclark/Documents/New project/nfl-big-data-bowl-2027


## 1. Locate or download the competition files

If you already ran the download snippet, paste its printed path into `EXISTING_DATA_DIR`. Otherwise leave it as `None` to use KaggleHub. Kaggle access may require signing in and accepting the rules on the competition page. Never put a token in this notebook.


In [2]:
DATA_DIR = Path(r"/Users/hunterclark/Documents/New project/nfl-big-data-bowl-2027/data/kagglehub/competitions/nfl-big-data-bowl-2027").expanduser().resolve()

if not DATA_DIR.is_dir():
    raise FileNotFoundError(DATA_DIR)

print("Competition files:", DATA_DIR)
print("CSV files found:", len(list(DATA_DIR.rglob("*.csv"))))


Competition files: /Users/hunterclark/Documents/New project/nfl-big-data-bowl-2027/data/kagglehub/competitions/nfl-big-data-bowl-2027
CSV files found: 9


## 2. Inventory every file

This lists filenames and disk size without loading tracking tables. CSV and compressed CSV files will be profiled. Other file formats remain in the inventory and need a separate reader if present.


In [3]:
inventory = pd.DataFrame(audit_data.inventory_files(DATA_DIR))

if inventory.empty:
    
    raise ValueError("No files found in DATA_DIR")

inventory["size_mb"] = (inventory["bytes"] / 1024**2).round(2)

display(inventory[["file", "size_mb", "is_csv"]])

inventory.to_csv(OUTPUT / "file_inventory.csv", index=False)

print("Total size (GB):", round(inventory["bytes"].sum() / 1024**3, 3))


,file,size_mb,is_csv
0,.DS_Store,0.01,False
1,nfl-big-data-bowl-2027/combine_results.csv,0.04,True
2,nfl-big-data-bowl-2027/combine_tracking.csv,79.91,True
3,nfl-big-data-bowl-2027/game_tracking_2023.csv,319.92,True
4,nfl-big-data-bowl-2027/game_tracking_2024.csv,684.69,True
5,nfl-big-data-bowl-2027/game_tracking_2025.csv,975.20,True
6,nfl-big-data-bowl-2027/games.csv,0.04,True
7,nfl-big-data-bowl-2027/player_career_successes...,0.01,True
8,nfl-big-data-bowl-2027/player_play.csv,140.07,True
9,nfl-big-data-bowl-2027/players.csv,0.05,True


Total size (GB): 2.148


## 3. Inspect columns, missingness, and numeric ranges

Start with the first 10,000 rows of each CSV. **These are leading samples, not random or representative samples.** Their missingness and ranges must not be reported as full-dataset results.

Set `FULL_SCAN = True` for exact row counts and full-file summaries. This streams chunks instead of loading the game tracking into memory. It may take several minutes. The audit treats pandas' standard missing-value tokens as missing; check identifier semantics before using that convention in analysis. IDs stay strings so leading zeros are preserved.


In [4]:
FULL_SCAN = False
SAMPLE_ROWS = 10_000

manifest = audit_data.audit(DATA_DIR, sample_rows=None if FULL_SCAN else SAMPLE_ROWS)
manifest_path = OUTPUT / ("data_audit_full.json" if FULL_SCAN else "data_audit_sample.json")
manifest_path.write_text(json.dumps(manifest, indent=2, allow_nan=False) + "\n")

file_rows, column_rows = [], []
for item in manifest["files"]:
    
    if not item["is_csv"]:
        continue
    
    file_rows.append({"file": item["file"], "rows_scanned": item.get("rows_scanned"),
                      "scope": item.get("scan_scope"), "error": item.get("error")})
    
    for column in item.get("columns", []):
        column_rows.append({"file": item["file"], **column})

file_summary = pd.DataFrame(file_rows)
column_summary = pd.DataFrame(column_rows)

display(file_summary)

if not column_summary.empty:
    display(column_summary.drop(columns=["examples"]))
    column_summary.to_csv(OUTPUT / "column_summary.csv", index=False)
print("Saved:", manifest_path)


,file,rows_scanned,scope,error
0,nfl-big-data-bowl-2027/combine_results.csv,510,full file,None
1,nfl-big-data-bowl-2027/combine_tracking.csv,10000,leading sample (may include entire file),None
2,nfl-big-data-bowl-2027/game_tracking_2023.csv,10000,leading sample (may include entire file),None
3,nfl-big-data-bowl-2027/game_tracking_2024.csv,10000,leading sample (may include entire file),None
4,nfl-big-data-bowl-2027/game_tracking_2025.csv,10000,leading sample (may include entire file),None
5,nfl-big-data-bowl-2027/games.csv,1002,full file,None
6,nfl-big-data-bowl-2027/player_career_successes...,510,full file,None
7,nfl-big-data-bowl-2027/player_play.csv,10000,leading sample (may include entire file),None
8,nfl-big-data-bowl-2027/players.csv,510,full file,None


,file,column,missing,numeric_values,nonfinite_numeric,numeric_min,numeric_max,missing_pct,numeric_parse_pct_nonmissing
0,nfl-big-data-bowl-2027/combine_results.csv,draft_year,0,510,0,2023.000,2025.000,0.000,100.0
1,nfl-big-data-bowl-2027/combine_results.csv,nfl_id,0,510,0,55867.000,59278.000,0.000,100.0
2,nfl-big-data-bowl-2027/combine_results.csv,combine_position,0,0,0,NaN,NaN,0.000,0.0
3,nfl-big-data-bowl-2027/combine_results.csv,combine_height,0,510,0,68.125,80.625,0.000,100.0
4,nfl-big-data-bowl-2027/combine_results.csv,combine_weight,0,510,0,154.000,374.000,0.000,100.0
...,...,...,...,...,...,...,...,...,...
153,nfl-big-data-bowl-2027/players.csv,college_name,0,0,0,NaN,NaN,0.000,0.0
154,nfl-big-data-bowl-2027/players.csv,college_conference,0,0,0,NaN,NaN,0.000,0.0
155,nfl-big-data-bowl-2027/players.csv,draft_round,127,383,0,1.000,7.000,24.902,100.0
156,nfl-big-data-bowl-2027/players.csv,draft_pick_within_round,127,383,0,1.000,43.000,24.902,100.0


Saved: /Users/hunterclark/Documents/New project/nfl-big-data-bowl-2027/outputs/audit/data_audit_sample.json


## 4. Preview the actual values and drill names

The previews below show three rows per file. Numeric-looking columns are not automatically measurements: a player ID can parse as a number too. Use the raw data dictionary to establish units and joins.


In [5]:
for item in manifest["files"]:
    
    if item["is_csv"] and "error" not in item and item.get("columns"):
        print("\n", item["file"])
        display(pd.read_csv(DATA_DIR / item["file"], dtype="string", nrows=3))



 nfl-big-data-bowl-2027/combine_results.csv


,draft_year,nfl_id,combine_position,combine_height,combine_weight,hand_size,arm_length,wing_span,ten_yd_split,forty,vertical,broad_jump,three_cone,short_shuttle,bench_reps,ngs_athleticism_score,ngs_college_production_score,ngs_final_score
0,2023,56040,TE,77.875,245,10,32.25,79.75,1.6,4.84,38.5,125,<NA>,<NA>,<NA>,60,<NA>,<NA>
1,2023,56010,DB,70.625,191,8.75,29.625,72.25,1.58,4.59,33.5,116,<NA>,<NA>,23,60,77,74
2,2023,56085,TE,79.375,255,10.25,34,83.125,1.57,4.55,40,128,6.87,4.12,23,97,65,79



 nfl-big-data-bowl-2027/combine_tracking.csv


,draft_year,event_id,nfl_id,entity_type,time,drill_type,drill_name,attempt,x,y,s,a,dis,dir
0,2025,20250302100900_1740941802777_000,58286,PLAYER,2025-03-02T18:56:38.200,SKILL_DRILLS_OL,OL_PULL_DRILL_FOLD_BLOCK_RIGHT,4,44.78592428,28.28552755,0.220828156,0.682442751,0.011085185,250.2785762
1,2025,20250302100900_1740941802777_000,58286,PLAYER,2025-03-02T18:56:38.300,SKILL_DRILLS_OL,OL_PULL_DRILL_FOLD_BLOCK_RIGHT,4,44.72378031,28.29739126,0.332931499,0.619550768,0.003864436,257.3192055
2,2025,20250302100900_1740941802777_000,58286,PLAYER,2025-03-02T18:56:38.400,SKILL_DRILLS_OL,OL_PULL_DRILL_FOLD_BLOCK_RIGHT,4,44.63268026,28.2795283,0.481838226,0.596455317,0.008305352,253.1768126



 nfl-big-data-bowl-2027/game_tracking_2023.csv


,game_id,play_id,nfl_id,time,x,y,s,a,dis,o,dir,event
0,2024010707,570,56450,2024-01-07T21:46:07.700,42.17,25.29,0,0,0,290.97,213.26,<NA>
1,2024010707,570,56450,2024-01-07T21:46:07.800,42.17,25.29,0,0,0,290.97,210.36,<NA>
2,2024010707,570,56450,2024-01-07T21:46:07.900,42.18,25.29,0,0,0,290.97,205.45,<NA>


... (8 outputs left out)


 nfl-big-data-bowl-2027/player_play.csv


,game_id,play_id,team_abbr,nfl_id,lined_up_position,play_description,quarter,down,yards_to_go,possession_team,...,sack,tackle,assist,caused_forced_fumble,recovered_fumble,tackle_for_loss,time_to_pressure,time_to_qb_hurry,coverage_assignment,coverage_assignment_at_snap
0,2025111610,2279,BAL,57183,T,(8:42) (Shotgun) L.Jackson pass incomplete dee...,3,1,10,BAL,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,2026011701,1626,SF,58248,INTERIOR_LINE,(3:48) (Shotgun) K.Walker left end to SEA 28 f...,2,2,7,SEA,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,2025101900,2838,LAR,57140,EDGE,(7:01) C.Van Lanen reported in as eligible. T...,3,2,2,JAX,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NO_ASSIGNMENT,NO_ASSIGNMENT



 nfl-big-data-bowl-2027/players.csv


,nfl_id,display_name,draft_year,nfl_position,birth_date,college_name,college_conference,draft_round,draft_pick_within_round,draft_overall_pick
0,55867,Will Anderson,2023,DE,2001-09-02,Alabama,Southeastern Conference,1,3,3
1,55870,Paris Johnson,2023,T,2001-07-31,Ohio State,Big Ten Conference,1,6,6
2,55874,Darnell Wright,2023,T,2001-08-10,Tennessee,Southeastern Conference,1,10,10


## 5. Confirm row keys and player-table coverage

After reviewing the actual columns, fill in `KEY_CHECKS` with small tables and their documented keys. Leave multi-million-row game tracking out of this in-memory check. Each configured table is read in full. Duplicate *rows involved* differs from the number of extra copies.

Example shape only (not verified column names): `{"players.csv": ["player_id"]}`.


In [6]:
KEY_CHECKS = {
    "nfl-big-data-bowl-2027/players.csv": ["nfl_id"],
    "nfl-big-data-bowl-2027/combine_results.csv": [
        "draft_year",
        "nfl_id",
    ],
    "nfl-big-data-bowl-2027/player_career_successes.csv": ["nfl_id"],
    "nfl-big-data-bowl-2027/games.csv": ["game_id"],
    "nfl-big-data-bowl-2027/player_play.csv": [
        "game_id",
        "play_id",
        "nfl_id",
    ],
}  # Map relative CSV filename -> documented key column(s).

key_results = []

for filename, keys in KEY_CHECKS.items():
    table = pd.read_csv(DATA_DIR / filename, dtype="string")
    absent = set(keys) - set(table.columns)
    
    if not keys or absent:
        
        raise ValueError(f"Invalid key mapping for {filename}: {keys}; absent={absent}")
    
    key_results.append({"file": filename, "rows": len(table), "keys": ", ".join(keys),
                        "rows_with_missing_key": int(table[keys].isna().any(axis=1).sum()),
                        "rows_in_duplicate_keys": int(table.duplicated(keys, keep=False).sum()),
                        "distinct_complete_keys": len(table[keys].dropna().drop_duplicates())})
if key_results:
    key_summary = pd.DataFrame(key_results)
    display(key_summary)
    key_summary.to_csv(OUTPUT / "key_checks.csv", index=False)

else:
    print("Pending: confirm the player and results table keys from the preview/dictionary.")


,file,rows,keys,rows_with_missing_key,rows_in_duplicate_keys,distinct_complete_keys
0,nfl-big-data-bowl-2027/players.csv,510,nfl_id,0,0,510
1,nfl-big-data-bowl-2027/combine_results.csv,510,"draft_year, nfl_id",0,0,510
2,nfl-big-data-bowl-2027/player_career_successes...,510,nfl_id,0,0,510
3,nfl-big-data-bowl-2027/games.csv,1002,game_id,0,0,1002
4,nfl-big-data-bowl-2027/player_play.csv,314197,"game_id, play_id, nfl_id",0,0,314197


## 6. Configure the Combine tracking audit

Use the actual filename and column names printed above. Do not guess attempt keys: an attempt ID may repeat across players, drills, or years. Include every column needed to distinguish attempts in `GROUP_COLS`.

First configure just `COMBINE_FILE` and `DRILL_COL` and run the next cell to see drill labels. Then set `FORTY_VALUES` to the exact 40-yard label(s). This reads the Combine file into memory, **not** the NFL game tracking files.


In [7]:
# Find the Combine tracking file inside your downloaded folder.
matches = list(DATA_DIR.rglob("combine_tracking.csv"))

if len(matches) != 1:
    raise ValueError(f"Expected one Combine tracking file; found {len(matches)}")

COMBINE_FILE = str(matches[0].relative_to(DATA_DIR))

DRILL_COL = "drill_name"
PLAYER_COL = "nfl_id"

GROUP_COLS = [
    "draft_year",
    "event_id",
    "nfl_id",
    "entity_type",
    "drill_name",
    "attempt",
]

TIME_COL = "time"
TIME_FORMAT = "datetime"
TIME_SCALE = None  # Not needed for datetime timestamps

SPEED_COL = "s"
ACCELERATION_COL = "a"

# Leave these unchanged until we inspect labels and confirm units.
FORTY_VALUES = ["FORTY_YARD_DASH"]
UNITS_CONFIRMED = False

combine = pd.read_csv(
    DATA_DIR / COMBINE_FILE,
    dtype="string",
)

print("Combine tracking rows:", len(combine))

# Counts represent recorded frames, not players or attempts.
drill_counts = (
    combine.groupby(
        ["drill_type", "drill_name"],
        dropna=False,
    )
    .size()
    .rename("frames")
    .reset_index()
)

display(drill_counts)
print("Entity types:")
display(combine["entity_type"].value_counts(dropna=False))


Combine tracking rows: 463189


,drill_type,drill_name,frames
0,FORTY_YARD_DASH,FORTY_YARD_DASH,41500
1,SHORT_SHUTTLE,SHORT_SHUTTLE,11690
2,SKILL_DRILLS_DB,BACK_PEDAL_AND_90_DEGREE_BREAK_TO_BALL,8422
3,SKILL_DRILLS_DB,BACK_PEDAL_AND_TRANSITION_45_DEGREE_REACTION,21569
4,SKILL_DRILLS_DB,BOX_DRILL,9439
...,...,...,...
57,SKILL_DRILLS_WR,SLOT_SAIL_ROUTE_RIGHT,9524
58,SKILL_DRILLS_WR,SLOT_STRIKE_ROUTE_LEFT,8457
59,SKILL_DRILLS_WR,SPEED_OUT_LEFT,1675
60,SKILL_DRILLS_WR,SPEED_OUT_ROUTE_LEFT,5619


Entity types:


entity_type
PLAYER    431094
BALL       32095
Name: count, dtype: Int64

## 7. Check attempts, duplicate timestamps, cadence, and first observed speed

Only selected 40-yard attempts enter this diagnostic. It reports missing keys/times, duplicate timestamps, observed duration, sampling cadence, unusually large gaps, and first/last/peak observed speed. Acceleration extrema are raw observations, not a validated peak-acceleration feature.

A high first observed speed is a reason to investigate a missed launch. A low first observed speed alone does not prove start coverage. Sorting timestamps allows gap checks but cannot reveal acquisition-order reversals. Finish coverage still requires spatial inspection in notebook 02.


In [8]:
attempt_summary = pd.DataFrame()
quality = {}
ready = (
    combine is not None
    and bool(DRILL_COL)
    and bool(FORTY_VALUES)
    and bool(GROUP_COLS)
    and bool(TIME_COL)
    and (TIME_FORMAT == "datetime" or TIME_SCALE is not None)
)

if ready:
    forties = combine.loc[
    combine[DRILL_COL].isin(FORTY_VALUES)
    & combine["entity_type"].eq("PLAYER")
].copy()
    
    if forties.empty:
        
        raise ValueError("No records matched FORTY_VALUES; inspect the drill frequency table.")
    
    if TIME_FORMAT == "datetime":
        forties[TIME_COL] = pd.to_datetime(forties[TIME_COL], errors="coerce", utc=True)
    
    elif TIME_FORMAT != "numeric":
        
        raise ValueError("TIME_FORMAT must be numeric or datetime")
    
    attempt_summary, quality = audit_data.tracking_diagnostics(
    forties,
    GROUP_COLS,
    TIME_COL,
    speed_col=SPEED_COL if UNITS_CONFIRMED else None,
    acceleration_col=ACCELERATION_COL if UNITS_CONFIRMED else None,
    time_scale=TIME_SCALE if TIME_FORMAT == "numeric" else 1.0,
    )
    
    display(pd.Series(quality, name="Value").to_frame())
    display(attempt_summary.head(20))
    
    attempt_summary.to_csv(OUTPUT / "forty_attempt_audit.csv", index=False)
    (OUTPUT / "tracking_quality.json").write_text(json.dumps(quality, indent=2) + "\n")
    
    if not attempt_summary.empty:
        display(attempt_summary.describe(include="all").T)
        
        if PLAYER_COL and PLAYER_COL in attempt_summary:
            counts = attempt_summary.groupby(PLAYER_COL, dropna=False).size()
            
            print("Players:", len(counts), "Players with multiple attempts:", int((counts > 1).sum()))

else:
    print("Tracking audit pending: finish the explicit column/units configuration above.")


,Value
input_rows,41500
invalid_time_rows,0
missing_group_key_rows,0
excluded_rows,0
rows_in_duplicate_timestamps,0
start_coverage,UNVERIFIED: first observed frame is not necess...
finish_coverage,UNVERIFIED: confirm spatial coverage separately
speed_units,raw units; verify against data dictionary


,draft_year,event_id,nfl_id,entity_type,drill_name,attempt,n_frames,n_unique_timestamps,observed_duration_s,median_step_s,estimated_hz,max_gap_s,gaps_over_1_5x_median,duplicate_timestamp_rows
0,2025,20250302101000_1740946089384_000,58896,PLAYER,FORTY_YARD_DASH,1,59,59,5.8,0.1,10.00001,0.1,0,0
1,2025,20250302101000_1740946604458_000,58896,PLAYER,FORTY_YARD_DASH,2,56,56,5.5,0.1,10.00001,0.1,0,0
2,2025,20250302101000_1740946012561_000,58261,PLAYER,FORTY_YARD_DASH,1,61,61,6.0,0.1,10.00001,0.1,0,0
3,2025,20250302101000_1740946523648_000,58261,PLAYER,FORTY_YARD_DASH,2,59,59,5.8,0.1,10.00001,0.1,0,0
4,2025,20250302100900_1740939268259_000,58454,PLAYER,FORTY_YARD_DASH,1,58,58,5.7,0.1,10.00001,0.1,0,0
5,2025,20250302100900_1740939859215_000,58454,PLAYER,FORTY_YARD_DASH,2,58,58,5.7,0.1,10.00001,0.1,0,0
6,2025,20250302100900_1740939746431_000,58259,PLAYER,FORTY_YARD_DASH,2,56,56,5.5,0.1,10.00001,0.1,0,0
7,2025,20250302100900_1740939115666_000,58259,PLAYER,FORTY_YARD_DASH,1,60,60,5.9,0.1,10.00001,0.1,0,0
8,2025,20250302101000_1740946761507_000,58300,PLAYER,FORTY_YARD_DASH,2,52,52,5.1,0.1,10.00001,0.1,0,0
9,2025,20250302101000_1740946276215_000,58300,PLAYER,FORTY_YARD_DASH,1,54,54,5.3,0.1,10.00001,0.1,0,0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
draft_year,793,3,2024,269,NaN,NaN,NaN,NaN,NaN,NaN,NaN
event_id,793,793,20250302101000_1740946089384_000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nfl_id,793,418,58896,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
entity_type,793,1,PLAYER,793,NaN,NaN,NaN,NaN,NaN,NaN,NaN
drill_name,793,1,FORTY_YARD_DASH,793,NaN,NaN,NaN,NaN,NaN,NaN,NaN
attempt,793,4,1,372,NaN,NaN,NaN,NaN,NaN,NaN,NaN
n_frames,793.0,NaN,NaN,NaN,52.332913,3.592942,45.0,50.0,51.0,55.0,90.0
n_unique_timestamps,793.0,NaN,NaN,NaN,52.332913,3.592942,45.0,50.0,51.0,55.0,90.0
observed_duration_s,793.0,NaN,NaN,NaN,5.133291,0.359294,4.4,4.9,5.0,5.4,8.9
median_step_s,793.0,NaN,NaN,NaN,0.1,0.0,0.1,0.1,0.1,0.1,0.1


Players: 418 Players with multiple attempts: 375


In [9]:
attempts_per_player = (
    attempt_summary.groupby("nfl_id")
    .size()
    .rename("recorded_attempts")
)

display(
    attempts_per_player.value_counts()
    .sort_index()
    .rename_axis("attempts_per_player")
    .to_frame("players")
)

# Inspect unusually long recordings.
display(
    attempt_summary.nlargest(10, "observed_duration_s")[
        ["draft_year", "nfl_id", "event_id",
         "attempt", "n_frames", "observed_duration_s"]
    ]
)

,players
attempts_per_player,
1,43
2,375


,draft_year,nfl_id,event_id,attempt,n_frames,observed_duration_s
637,2023,55999,64066606e3d8e72dcd850840-2023,4,90,8.9
312,2024,57207,20240303101000_1709497471519_000,1,63,6.2
348,2024,57164,20240301100300_1709324534663_000,1,63,6.2
47,2025,58987,20250302101000_1740945931547_000,1,62,6.1
302,2024,57293,20240303100900_1709490621288_000,1,62,6.1
585,2023,56450,6406869fe3d8e72dcdd1b902-2023,2,62,6.1
2,2025,58261,20250302101000_1740946012561_000,1,61,6.0
174,2025,58333,20250301100800_1740870489242_000,1,61,6.0
298,2024,57353,20240303101000_1709497664752_000,1,61,6.0
314,2024,57335,20240303101000_1709496779458_000,1,61,6.0


In [10]:
players = pd.read_csv(
    DATA_DIR / "nfl-big-data-bowl-2027/players.csv",
    dtype="string",
)

# One row per player with recorded 40-yard tracking.
tracked_players = (
    attempt_summary.groupby("nfl_id")
    .size()
    .rename("recorded_attempts")
    .reset_index()
)

cohort = tracked_players.merge(
    players[
        ["nfl_id", "display_name", "draft_year", "nfl_position"]
    ],
    on="nfl_id",
    how="left",
    validate="one_to_one",
    indicator=True,
)

print("Tracking players without a player-table match:")
display(cohort.loc[cohort["_merge"] != "both"])

print("Tracked players by position and draft year:")
display(pd.crosstab(cohort["nfl_position"], cohort["draft_year"]))

wr_cohort = cohort.loc[cohort["nfl_position"].eq("WR")].copy()

print("Wide receivers with tracking:", len(wr_cohort))
print(
    "Wide receivers with two recorded attempts:",
    wr_cohort["recorded_attempts"].eq(2).sum(),
)

Tracking players without a player-table match:


,nfl_id,recorded_attempts,display_name,draft_year,nfl_position,_merge


Tracked players by position and draft year:


draft_year,2023,2024,2025
nfl_position,,,
C,6,8,2
CB,14,22,15
DB,0,1,2
DE,13,10,7
DT,8,15,14
FB,0,0,1
FS,6,9,6
G,14,12,14
ILB,0,1,0


Wide receivers with tracking: 93
Wide receivers with two recorded attempts: 81


## 8. Record decisions before feature extraction

Edit the decision log below after inspecting the output. Unresolved items must remain marked pending. The profile files and decision log will be the handoff to notebook 02. Do not report synthetic tests or leading-sample diagnostics as competition findings.


In [11]:
decisions = {
    "analysis_population": (
        "Proposed WR cohort: 93 players with recorded 40 tracking; "
        "32 from 2023, 28 from 2024, 33 from 2025. "
        "Final eligibility pending trajectory and NFL outcome checks."
    ),
    "raw_columns_and_units": (
        "Column names identified; speed and acceleration units "
        "and definitions pending dictionary review."
    ),
    "player_and_attempt_keys": (
        "Five tested tables have complete, unique candidate keys. "
        "No duplicate timestamps within configured tracking-attempt groups."
    ),
    "sampling_frequency": (
        "Consistent approximately 10 Hz across 793 attempts; "
        "no sampling gaps detected."
    ),
    "physical_launch_coverage": "Pending trajectory inspection.",
    "full_40_finish_coverage": "Pending spatial inspection.",
    "acceleration_definition": "Pending signed-versus-magnitude verification.",
    "repeat_attempt_handling": (
        "Retain all recordings for now. "
        "WR cohort: 81 players with two attempts, 12 with one."
    ),
    "combine_to_nfl_join": (
        "All 418 tracked players match players.csv. "
        "NFL game-tracking and player-play overlap still pending."
    ),
    "game_outcome_availability": "Pending route/event and exposure audit.",
    "unusual_recordings": (
        "One 8.9-second recording flagged for inspection; not excluded."
    ),
    "audit_scope": (
        "Full selected 40-yard tracking and key checks; "
        "other CSV profiling follows FULL_SCAN setting."
    ),
}

(OUTPUT / "decisions.json").write_text(json.dumps(decisions, indent=2) + "\n")
display(pd.Series(decisions, name="Decision").to_frame())
print("Audit exports:", OUTPUT)


,Decision
analysis_population,Proposed WR cohort: 93 players with recorded 4...
raw_columns_and_units,Column names identified; speed and acceleratio...
player_and_attempt_keys,"Five tested tables have complete, unique candi..."
sampling_frequency,Consistent approximately 10 Hz across 793 atte...
physical_launch_coverage,Pending trajectory inspection.
full_40_finish_coverage,Pending spatial inspection.
acceleration_definition,Pending signed-versus-magnitude verification.
repeat_attempt_handling,Retain all recordings for now. WR cohort: 81 p...
combine_to_nfl_join,All 418 tracked players match players.csv. NFL...
game_outcome_availability,Pending route/event and exposure audit.


Audit exports: /Users/hunterclark/Documents/New project/nfl-big-data-bowl-2027/outputs/audit


## Handoff checklist

- [x] File inventory and actual column names inspected.
- [x] Sample versus full-file statistics labeled.
- [x] 40-yard drill label identified and player records selected.
- [x] Candidate table keys checked; no missing or duplicate keys found.
- [x] Tracking timestamps checked: approximately 10 Hz, with no detected gaps.
- [x] All 418 tracked players matched to the player table.
- [x] Proposed WR cohort identified: 93 players, including 81 with two attempts.
- [ ] Speed and acceleration units and definitions verified.
- [ ] Physical launch and full-40 finish coverage inspected.
- [ ] Unusually long recordings inspected.
- [ ] NFL tracking and player-play overlap, route context, and opportunities audited.

No recordings have been excluded based on duration alone.

Notebook 02 will inspect trajectories and resolve launch/finish coverage
before calculating acceleration features. Unresolved definitions remain
documented as pending.

Local execution uses the downloaded competition files; KaggleHub
authentication is not required.